# LAP × 联想记忆：容量与干预鲁棒性

这个 notebook 会完成链式 SCM 与混杂 fork 的数据生成、Modern Hopfield 对照、LAP-regularized E-SCM 训练、do-intervention、容量/吸引域评估和四张图。

> 默认配置是端到端烟雾实验。确认它能完整跑完后，再打开完整扫描。

## 1. 获取代码与依赖

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/Heptazero/nn-labs.git"
REVISION = "agent/add-lap-associative-memory-colab"
REPO_DIR = "/content/nn-labs"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", REVISION, "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("Repository ready:", os.getcwd())

## 2. 选择实验规模

`False` 通常几分钟内完成，用于检查实现；`True` 是多 seed 扫描，二阶导开销明显更高。

In [ ]:
from pathlib import Path
import torch
from lap_associative_memory import ExperimentConfig, plot_results, run_grid

FULL_EXPERIMENT = False
config = ExperimentConfig.full() if FULL_EXPERIMENT else ExperimentConfig()
OUTPUT_DIR = Path("/content/lap_results/full" if FULL_EXPERIMENT else "/content/lap_results/quick")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(config)
print("device =", device, "| output =", OUTPUT_DIR)

如果要防止 Colab 断连丢失结果，可先挂载 Drive，然后把 `OUTPUT_DIR` 改成 `/content/drive/MyDrive/lap_results/...`。

In [ ]:
# 可选：保存到 Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# OUTPUT_DIR = Path('/content/drive/MyDrive/lap_results/full')

## 3. 训练、评估与 checkpoint

In [ ]:
metrics, history = run_grid(config, OUTPUT_DIR, device=device)
figure_paths = plot_results(metrics, history, OUTPUT_DIR)
print(f"完成：{len(metrics)} 条指标，{len(history)} 条训练记录")
display(metrics.head(12))

## 4. 核心汇总

In [ ]:
summary = (metrics
           .groupby(['model', 'lambda_lap', 'evaluation', 'metric'], dropna=False)['value']
           .agg(['mean', 'std', 'count'])
           .reset_index())
display(summary)

## 5. 四张实验图

In [ ]:
from IPython.display import Image, display
for path in figure_paths:
    print(path.name)
    display(Image(filename=str(path)))

## 读图边界

先确认图 4 中 λ>0 的 LAP penalty 确实下降，再解释泄漏或容量差异。烟雾配置只验证流水线，不能作为论文结论；正式结果至少使用完整配置、置信区间，并检查不同检索步数下结论是否稳定。